# EN4554 Assignment 1, Part 2: a tool-calling agent, before and after LoRA

**This notebook**:
1. Builds a minimal agent around **Qwen2.5-0.5B-Instruct** with 4 tools that use made-up data.
2. Generates a training set and a held-out test set of **100 requests** whose phrasings never appear in training.
3. Evaluates the untouched model in the agent loop, once with the plain prompt and once with a stronger hand-written prompt.
4. LoRA-finetunes the model (loss on the assistant turns only) and evaluates it again on the same test set.
5. Reports **call validity, call accuracy, abstention and task success**, and analyses the failures.



In [1]:
!pip install -q "transformers==5.19.0" "peft==0.21.2" accelerate
!pip uninstall -y -q torchao

In [2]:
import os, sys, re, json, time, math, random, datetime, collections, gc
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU memory fragmentation
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/EN4554_Agent"
else:
    ROOT = "./EN4554_Agent"
DATA_DIR, RES_DIR, ADAPTER_DIR = f"{ROOT}/data", f"{ROOT}/results", f"{ROOT}/lora_adapter"
for d in (DATA_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"          # fp16 autocast on the T4 (no bf16 on a T4)

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ---------------- settings----------------
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_STEPS = 4                 # agent loop: at most 4 model turns per request
MAX_NEW_TOKENS = 256          # per model turn, greedy decoding
EVAL_BATCH = 16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LR, EPOCHS, TRAIN_BATCH, GRAD_ACCUM = 2e-4, 2, 2, 4      # effective batch 8 (2 conversations x 4 accumulation steps)
WARMUP_FRAC, MAX_GRAD_NORM, MAX_LEN = 0.05, 1.0, 1024
EVAL_LIMIT = None             # None = all 100 test requests (set a small number only for a quick test)
MAX_TRAIN_STEPS = None        # None = full training
RERUN = False                 # True = ignore saved results on Drive and recompute

import transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

Mounted at /content/drive
torch 2.14.1+cu130 | transformers 5.19.0 | peft 0.21.2 | device cuda Tesla T4


## 1. The tools
Four simple Python functions. The weather, calendar and contacts use **made-up data**, so the model cannot answer from memory: it has to call the tool. The unit converter computes exact answers that a 0.5B model usually gets wrong in its head.

| Tool | Arguments | Returns |
|---|---|---|
| `get_weather` | `city`, optional `unit` (celsius / fahrenheit) | temperature, condition, humidity |
| `get_events` | `date` (YYYY-MM-DD) | the user's calendar events that day |
| `convert_units` | `value`, `from_unit`, `to_unit` | the converted value (2 decimals) |
| `find_contact` | `name` | phone, email and home city |

The agent is told that today is **Tuesday, 10 March 2026**, so "tomorrow" means 2026-03-11.

In [3]:
TODAY = datetime.date(2026, 3, 10)
SYSTEM_PROMPT = "You are a helpful personal assistant. Today is Tuesday, 10 March 2026."

# ---------------- made-up data ----------------
WEATHER = {   # city: (temperature in °C, condition, humidity %)
    "Colombo": (31, "partly cloudy", 78), "Kandy": (24, "light rain", 85), "Galle": (29, "thunderstorms", 88),
    "Jaffna": (33, "sunny", 62), "Nuwara Eliya": (16, "foggy", 92), "London": (9, "overcast", 81),
    "Paris": (12, "light rain", 76), "Tokyo": (14, "clear", 55), "New York": (6, "windy", 48),
    "Sydney": (24, "sunny", 60), "Dubai": (35, "sunny", 30), "Singapore": (30, "thunderstorms", 84),
    "Mumbai": (32, "hazy", 70), "Chennai": (34, "partly cloudy", 66), "Berlin": (3, "snow", 87),
    "Toronto": (1, "snow", 72), "Cape Town": (22, "windy", 58), "Bangkok": (34, "clear", 65),
}
COND_REGEX = {"partly cloudy": r"cloud", "light rain": r"rain", "thunderstorms": r"thunder|storm", "sunny": r"sun",
              "foggy": r"fog", "overcast": r"overcast|cloud", "clear": r"clear", "windy": r"wind", "hazy": r"haz", "snow": r"snow"}

EVENTS = {    # date: [(time, title, pattern a correct answer must contain)]
    "2026-03-10": [("09:30", "Team stand-up", r"stand-?up"), ("15:00", "Dentist appointment", r"dentist")],
    "2026-03-11": [("11:00", "Project review with Sarah", r"review")],
    "2026-03-12": [("18:30", "Badminton with Kamal", r"badminton")],
    "2026-03-13": [("10:00", "Quarterly planning meeting", r"planning"), ("13:00", "Lunch with Aiko", r"lunch")],
    "2026-03-14": [("08:00", "Morning run", r"\brun\b"), ("19:00", "Dinner at Nimal's place", r"dinner")],
    "2026-03-16": [("14:00", "Car service", r"\bcar\b")],
    "2026-03-17": [("09:00", "Flight to Singapore", r"flight")],
    "2026-03-19": [("16:00", "Guitar lesson", r"guitar")],
    "2026-03-20": [("12:00", "Doctor's check-up", r"doctor|check-?up"), ("17:00", "Book club", r"\bbook")],
    "2026-03-23": [("10:30", "Job interview panel", r"interview")],
    "2026-03-25": [("15:30", "Parent-teacher meeting", r"parent|teacher")],
    "2026-03-27": [("20:00", "Movie night", r"movie")],
    "2026-03-30": [("09:00", "Tax filing deadline", r"\btax")],
}

CONTACTS = [   # fictional people; phone numbers use the reserved 555 range, emails use example.com
    {"name": "Nimal Perera", "phone": "+94 77 555 4567", "email": "nimal.perera@example.com", "city": "Kandy"},
    {"name": "Sarah Collins", "phone": "+44 20 5550 0321", "email": "sarah.collins@example.com", "city": "London"},
    {"name": "Kamal Silva", "phone": "+94 71 555 0192", "email": "kamal.silva@example.com", "city": "Galle"},
    {"name": "Aiko Tanaka", "phone": "+81 3 5550 1234", "email": "aiko.tanaka@example.com", "city": "Tokyo"},
    {"name": "Ravi Kumar", "phone": "+91 98400 55512", "email": "ravi.kumar@example.com", "city": "Chennai"},
    {"name": "Emma Dubois", "phone": "+33 1 5550 7788", "email": "emma.dubois@example.com", "city": "Paris"},
    {"name": "Daniel Brooks", "phone": "+1 212 555 0147", "email": "daniel.brooks@example.com", "city": "New York"},
    {"name": "Fatima Al-Sayed", "phone": "+971 4 555 0199", "email": "fatima.alsayed@example.com", "city": "Dubai"},
    {"name": "Lukas Weber", "phone": "+49 30 5550 4411", "email": "lukas.weber@example.com", "city": "Berlin"},
    {"name": "Chloe Martin", "phone": "+1 416 555 0123", "email": "chloe.martin@example.com", "city": "Toronto"},
    {"name": "Tharindu Jayasuriya", "phone": "+94 76 555 8800", "email": "tharindu.j@example.com", "city": "Colombo"},
    {"name": "Priya Nair", "phone": "+91 22 5550 6677", "email": "priya.nair@example.com", "city": "Mumbai"},
    {"name": "Oliver Smith", "phone": "+61 2 5550 3322", "email": "oliver.smith@example.com", "city": "Sydney"},
    {"name": "Mei Lin", "phone": "+65 5550 9090", "email": "mei.lin@example.com", "city": "Singapore"},
    {"name": "Sipho Ndlovu", "phone": "+27 21 555 0456", "email": "sipho.ndlovu@example.com", "city": "Cape Town"},
    {"name": "Anjali Fernando", "phone": "+94 21 555 3030", "email": "anjali.fernando@example.com", "city": "Jaffna"},
]

UNITS = {     # canonical unit: (quantity, size in the base unit); temperatures are handled separately
    "kilometers": ("length", 1000.0), "meters": ("length", 1.0), "centimeters": ("length", 0.01),
    "miles": ("length", 1609.344), "feet": ("length", 0.3048), "inches": ("length", 0.0254),
    "kilograms": ("mass", 1.0), "grams": ("mass", 0.001), "pounds": ("mass", 0.45359237), "ounces": ("mass", 0.028349523125),
    "liters": ("volume", 1.0), "milliliters": ("volume", 0.001), "gallons": ("volume", 3.785411784),
    "celsius": ("temperature", None), "fahrenheit": ("temperature", None),
}
_UNIT_ALIASES = {"km": "kilometers", "kilometer": "kilometers", "kilometre": "kilometers", "kilometres": "kilometers",
                 "m": "meters", "meter": "meters", "metre": "meters", "metres": "meters",
                 "cm": "centimeters", "centimeter": "centimeters", "centimetre": "centimeters", "centimetres": "centimeters",
                 "mi": "miles", "mile": "miles", "ft": "feet", "foot": "feet", "in": "inches", "inch": "inches",
                 "kg": "kilograms", "kgs": "kilograms", "kilo": "kilograms", "kilos": "kilograms", "kilogram": "kilograms",
                 "g": "grams", "gram": "grams", "lb": "pounds", "lbs": "pounds", "pound": "pounds", "oz": "ounces", "ounce": "ounces",
                 "l": "liters", "liter": "liters", "litre": "liters", "litres": "liters",
                 "ml": "milliliters", "milliliter": "milliliters", "millilitre": "milliliters", "millilitres": "milliliters",
                 "gal": "gallons", "gallon": "gallons", "c": "celsius", "degreescelsius": "celsius", "centigrade": "celsius",
                 "f": "fahrenheit", "degreesfahrenheit": "fahrenheit"}

# ---------------- name normalisation (the tools accept small spelling variations) ----------------
def _key(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

_CITY_KEYS = {_key(c): c for c in WEATHER}
def canon_city(x):
    return _CITY_KEYS.get(_key(str(x).split(",")[0]), str(x).strip().lower())

def canon_temp_unit(u):
    k = _key(u).replace("degrees", "")
    return {"celsius": "celsius", "c": "celsius", "metric": "celsius",
            "fahrenheit": "fahrenheit", "f": "fahrenheit", "imperial": "fahrenheit"}.get(k, str(u).strip().lower())

def canon_unit(u):
    k = _key(u)
    return k if k in UNITS else _UNIT_ALIASES.get(k, str(u).strip().lower())

def find_contact_record(name):
    k = _key(name)
    for c in CONTACTS:
        parts = c["name"].split()
        if k and k in (_key(c["name"]), _key(parts[0]), _key(parts[-1])):
            return c
    return None

def canon_contact(name):
    c = find_contact_record(name)
    return c["name"] if c else str(name).strip().lower()

# ---------------- the four tools ----------------
def get_weather(city, unit="celsius"):
    c = canon_city(city)
    if c not in WEATHER:
        return {"error": f"No weather data for '{city}'."}
    u = canon_temp_unit(unit)
    if u not in ("celsius", "fahrenheit"):
        return {"error": f"Unknown unit '{unit}'. Use 'celsius' or 'fahrenheit'."}
    t, cond, hum = WEATHER[c]
    return {"city": c, "temperature": t if u == "celsius" else round(t * 9 / 5 + 32), "unit": u,
            "condition": cond, "humidity_percent": hum}

def get_events(date):
    d = str(date).strip()
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", d):
        return {"error": "The date must be in YYYY-MM-DD format."}
    return {"date": d, "events": [{"time": t, "title": title} for t, title, _ in EVENTS.get(d, [])]}

def convert_units(value, from_unit, to_unit):
    try:
        v = float(value)
    except (TypeError, ValueError):
        return {"error": f"'{value}' is not a number."}
    f, t = canon_unit(from_unit), canon_unit(to_unit)
    for u, raw in ((f, from_unit), (t, to_unit)):
        if u not in UNITS:
            return {"error": f"Unknown unit '{raw}'."}
    if UNITS[f][0] != UNITS[t][0]:
        return {"error": f"Cannot convert {f} to {t}."}
    if UNITS[f][0] == "temperature":
        r = v if f == t else (v * 9 / 5 + 32 if f == "celsius" else (v - 32) * 5 / 9)
    else:
        r = v * UNITS[f][1] / UNITS[t][1]
    return {"value": value, "from_unit": f, "to_unit": t, "result": round(r, 2)}

def find_contact(name):
    c = find_contact_record(name)
    return dict(c) if c else {"error": f"No contact named '{name}'."}

TOOL_FUNCS = {"get_weather": get_weather, "get_events": get_events, "convert_units": convert_units, "find_contact": find_contact}

# JSON schemas, given to the model through the chat template's `tools=` argument
TOOLS = [
    {"type": "function", "function": {
        "name": "get_weather", "description": "Get the current weather in a city: temperature, condition and humidity.",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "City name, e.g. 'London'"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Temperature unit (default: celsius)"}},
            "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "get_events", "description": "List the events in the user's calendar on one date.",
        "parameters": {"type": "object", "properties": {
            "date": {"type": "string", "description": "Date in YYYY-MM-DD format"}}, "required": ["date"]}}},
    {"type": "function", "function": {
        "name": "convert_units", "description": "Convert a value between units of length, mass, volume or temperature.",
        "parameters": {"type": "object", "properties": {
            "value": {"type": "number", "description": "The number to convert"},
            "from_unit": {"type": "string", "enum": list(UNITS), "description": "Unit of the value"},
            "to_unit": {"type": "string", "enum": list(UNITS), "description": "Unit to convert to"}},
            "required": ["value", "from_unit", "to_unit"]}}},
    {"type": "function", "function": {
        "name": "find_contact", "description": "Look up a person in the user's contacts: phone number, email and home city.",
        "parameters": {"type": "object", "properties": {
            "name": {"type": "string", "description": "The person's name"}}, "required": ["name"]}}},
]
_SCHEMA = {t["function"]["name"]: t["function"]["parameters"] for t in TOOLS}

def execute_call(name, args):
    if name not in TOOL_FUNCS:
        return {"error": f"Unknown tool '{name}'."}
    try:
        return TOOL_FUNCS[name](**args)
    except TypeError as e:
        return {"error": f"Bad arguments: {e}"}

def is_valid_call(c):
    # format check only: known tool, arguments object, required keys present, no unknown keys, correct JSON types
    if c.get("name") not in TOOL_FUNCS or not isinstance(c.get("arguments"), dict):
        return False
    schema, a = _SCHEMA[c["name"]], c["arguments"]
    if any(k not in a for k in schema["required"]) or any(k not in schema["properties"] for k in a):
        return False
    for k, v in a.items():
        want = schema["properties"][k]["type"]
        if want == "string" and not isinstance(v, str):
            return False
        if want == "number" and (isinstance(v, bool) or not isinstance(v, (int, float))):
            return False
    return True

def norm_call(c):
    # canonical form of a call, used to decide "right tool and right arguments"
    n, a = c.get("name"), c.get("arguments") or {}
    try:
        if n == "get_weather":
            return (n, canon_city(a.get("city")), canon_temp_unit(a.get("unit", "celsius")))
        if n == "get_events":
            return (n, str(a.get("date")).strip())
        if n == "convert_units":
            return (n, round(float(a.get("value")), 6), canon_unit(a.get("from_unit")), canon_unit(a.get("to_unit")))
        if n == "find_contact":
            return (n, canon_contact(a.get("name")))
    except (TypeError, ValueError):
        pass
    return (str(n), json.dumps(a, sort_keys=True))

for n, a in [("get_weather", {"city": "Kandy"}), ("get_events", {"date": "2026-03-11"}),
             ("convert_units", {"value": 10, "from_unit": "km", "to_unit": "miles"}), ("find_contact", {"name": "sarah"})]:
    print(f"{n}({a}) -> {execute_call(n, a)}")

get_weather({'city': 'Kandy'}) -> {'city': 'Kandy', 'temperature': 24, 'unit': 'celsius', 'condition': 'light rain', 'humidity_percent': 85}
get_events({'date': '2026-03-11'}) -> {'date': '2026-03-11', 'events': [{'time': '11:00', 'title': 'Project review with Sarah'}]}
convert_units({'value': 10, 'from_unit': 'km', 'to_unit': 'miles'}) -> {'value': 10, 'from_unit': 'kilometers', 'to_unit': 'miles', 'result': 6.21}
find_contact({'name': 'sarah'}) -> {'name': 'Sarah Collins', 'phone': '+44 20 5550 0321', 'email': 'sarah.collins@example.com', 'city': 'London'}


## 2. The agent loop
Goal → Plan → Act → Observe → Done:
1. The conversation (system prompt with the tool schemas, then the user request) is rendered with the model's chat template and the model **generates** a turn (greedy decoding).
2. Any `<tool_call>{...}</tool_call>` blocks are **parsed** as JSON.
3. If there are calls, each one is **executed** and its result is fed back as a `tool` message. Then the model generates again.
4. The loop **stops** when the model answers in plain text (no tool call), when a call cannot be parsed (format error), or after `MAX_STEPS` turns.

To save time, all unfinished conversations are advanced together in batches.

In [4]:
TOOL_CALL_RE = re.compile(r"<tool_call>(.*?)</tool_call>", re.S)

def parse_output(text):
    # returns (plain text, list of calls, format_error)
    blocks = TOOL_CALL_RE.findall(text)
    content = TOOL_CALL_RE.sub("", text).strip()
    if not blocks:
        # an attempted call that is not in the expected format (e.g. unclosed tag or bare JSON) is a format error
        attempted = "<tool_call>" in text or re.search(r'\{\s*"name"\s*:', text) is not None
        return content, [], attempted
    calls = []
    for b in blocks:
        try:
            c = json.loads(b.strip())
        except json.JSONDecodeError:
            return content, [], True
        if not (isinstance(c, dict) and isinstance(c.get("name"), str) and isinstance(c.get("arguments", {}), dict)):
            return content, [], True
        calls.append({"name": c["name"], "arguments": c.get("arguments", {})})
    return content, calls, False

@torch.no_grad()
def generate_batch(model, convs):
    tok.padding_side = "left"
    texts = [tok.apply_chat_template(c, tools=TOOLS, add_generation_prompt=True, tokenize=False) for c in convs]
    enc = tok(texts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
        out = model.generate(**enc, generation_config=GEN_CFG)
    return [tok.decode(o, skip_special_tokens=True).strip() for o in out[:, enc["input_ids"].shape[1]:]]

def run_agent(model, requests, system_prompt=SYSTEM_PROMPT, max_steps=MAX_STEPS, batch_size=EVAL_BATCH, verbose=False):
    model.eval()
    convs = [[{"role": "system", "content": system_prompt}, {"role": "user", "content": r}] for r in requests]
    traj = [{"request": r, "steps": [], "final": None, "stop": None} for r in requests]
    active = list(range(len(requests)))
    for step in range(max_steps):
        if not active:
            break
        outs = {}
        for i in range(0, len(active), batch_size):
            idx = active[i:i + batch_size]
            for k, o in zip(idx, generate_batch(model, [convs[k] for k in idx])):
                outs[k] = o
        still = []
        for k in active:
            content, calls, bad = parse_output(outs[k])
            rec = {"output": outs[k], "calls": calls, "format_error": bad, "results": []}
            traj[k]["steps"].append(rec)
            if bad:
                traj[k]["final"], traj[k]["stop"] = content or outs[k], "format_error"
            elif not calls:
                traj[k]["final"], traj[k]["stop"] = content, "answer"
            else:            # act + observe: run every call, feed the results back, continue the loop
                convs[k].append({"role": "assistant", "content": content,
                                 "tool_calls": [{"type": "function", "function": c} for c in calls]})
                for c in calls:
                    res = execute_call(c["name"], c["arguments"])
                    rec["results"].append(res)
                    convs[k].append({"role": "tool", "content": json.dumps(res)})
                still.append(k)
        active = still
        if verbose:
            print(f"   step {step + 1}: {len(active)} conversations still running", flush=True)
    for k in active:
        traj[k]["stop"] = "max_steps"
    return traj

def show(t):
    print("USER:", t["request"])
    for s in t["steps"]:
        print("  MODEL:", s["output"].replace("\n", " ")[:300])
        for r in s["results"]:
            print("  TOOL :", json.dumps(r)[:300])
    print(f"  -> stop: {t['stop']}\n")

## 3. Data
Requests are generated from templates with random cities, dates, values and people. Every example is a full conversation: user request → assistant tool call(s) → tool result(s) → assistant answer. Requests that need no tool get a direct answer.

* **Train and test use different templates**, so every test request is phrased differently from anything in training. Test dates are also written in different forms (e.g. "March 14th", "Mar 14" instead of "March 14", "14 March").
* **Categories:** a single call to each tool; **chains** (`find_contact` → `get_weather` for "the weather where Sarah lives"); **parallel** calls (two cities at once); and **no-tool** requests (general knowledge, arithmetic, chit-chat, and near-misses such as "How many seconds are in a minute?" or "What's the weather usually like in a desert?").
* Every test request stores what a correct final answer must contain: the temperature and condition, the event names, the converted number, the phone digits, the capital city, and so on.

In [5]:
def ordinal(n):
    return f"{n}{'th' if 11 <= n % 100 <= 13 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"

def human_date(iso):
    d = datetime.date.fromisoformat(iso)
    return f"{d.strftime('%A')}, {d.day} March"

def fmt_num(x):
    return f"{x:.2f}".rstrip("0").rstrip(".") if isinstance(x, float) else str(x)

def build(split, cat, request, steps, answer, check):
    # steps: list of (list of (tool, args), list of results); returns one example with its full gold conversation
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": request}]
    for calls, results in steps:
        msgs.append({"role": "assistant", "content": "",
                     "tool_calls": [{"type": "function", "function": {"name": n, "arguments": a}} for n, a in calls]})
        for r in results:
            msgs.append({"role": "tool", "content": json.dumps(r)})
    msgs.append({"role": "assistant", "content": answer})
    return {"split": split, "category": cat, "request": request, "needs_tool": bool(steps),
            "calls": [[{"name": n, "arguments": a} for n, a in calls] for calls, _ in steps],
            "check": check, "messages": msgs}

def run(_tool, **args):
    return (_tool, args), TOOL_FUNCS[_tool](**args)

# ---------------- weather ----------------
W_T = {"train": [("What's the weather in {city}?", "c", "tc"), ("How is the weather in {city} right now?", "c", "tc"),
                 ("Tell me the current weather for {city}.", "c", "tc"), ("Weather in {city} please", "c", "tc"),
                 ("Can you check the weather for {city}?", "c", "tc"), ("I'm heading to {city}. What's the weather there?", "c", "tc"),
                 ("What's the temperature in {city}?", "c", "t"), ("Is it hot in {city} today?", "c", "t"),
                 ("What's the weather like in {city} in Fahrenheit?", "f", "tc"), ("Give me the temperature in {city} in Fahrenheit.", "f", "t")],
       "test": [("Could you look up current conditions in {city}?", "c", "tc"), ("{city} weather?", "c", "tc"),
                ("Any idea what the weather's doing in {city}?", "c", "tc"), ("How warm is it in {city} at the moment?", "c", "t"),
                ("Check how cold it is in {city}.", "c", "t"), ("Do I need an umbrella in {city} today?", "c", "c"),
                ("Report the weather in {city} using Fahrenheit.", "f", "tc"), ("What's it like outside in {city} now, in degrees F?", "f", "tc")]}

def weather_answer(rng, r):
    u = "°C" if r["unit"] == "celsius" else "°F"
    return rng.choice([f"It's currently {r['temperature']}{u} and {r['condition']} in {r['city']}, with {r['humidity_percent']}% humidity.",
                       f"In {r['city']} it's {r['temperature']}{u} right now, {r['condition']}, humidity {r['humidity_percent']}%.",
                       f"The weather in {r['city']}: {r['condition']}, {r['temperature']}{u}, {r['humidity_percent']}% humidity."])

def weather_check(rs, need):
    spec = {"regex": [], "numbers": []}
    for r in rs:
        if "t" in need:
            spec["numbers"].append(r["temperature"])
        if "c" in need:
            spec["regex"].append([COND_REGEX[r["condition"]]])
    return spec

def gen_weather(rng, split):
    tpl, unit, need = rng.choice(W_T[split])
    city = rng.choice(list(WEATHER))
    call, r = run("get_weather", city=city) if unit == "c" else run("get_weather", city=city, unit="fahrenheit")
    return build(split, "weather", tpl.format(city=city), [([call], [r])], weather_answer(rng, r), weather_check([r], need))

# ---------------- calendar ----------------
C_T = {"train": ["What's on my calendar {on}?", "Do I have any events {on}?", "Show me my schedule {fr}.",
                 "What meetings do I have {on}?", "Am I busy {on}?", "List my appointments {on}.", "Check my calendar {fr}."],
       "test": ["Anything scheduled {on}?", "What does my day look like {on}?", "Have I got plans {on}?",
                "Pull up my agenda {fr}.", "Is my diary free {on}?", "Remind me what I've got going on {on}."]}
DATE_FORMS = {"train": ["March {d}", "{d} March", "the {o} of March", "{d}/03/2026"],
              "test": ["March {o}", "{o} March", "Mar {d}", "2026-03-{d:02d}"]}
EVENT_DAYS = [int(k[-2:]) for k in EVENTS if k > "2026-03-11"]
FREE_DAYS = [d for d in range(12, 32) if f"2026-03-{d:02d}" not in EVENTS]

def pick_date(rng, split):
    r = rng.random()
    if r < 0.15:
        return TODAY.isoformat(), "today", "for today"
    if r < 0.30:
        return (TODAY + datetime.timedelta(days=1)).isoformat(), "tomorrow", "for tomorrow"
    day = rng.choice(EVENT_DAYS) if rng.random() < 0.6 else rng.choice(FREE_DAYS)
    p = rng.choice(DATE_FORMS[split]).format(d=day, o=ordinal(day))
    return f"2026-03-{day:02d}", f"on {p}", f"for {p}"

def gen_calendar(rng, split):
    iso, on, fr = pick_date(rng, split)
    call, r = run("get_events", date=iso)
    hd = human_date(iso)
    if r["events"]:
        items = [f"{e['title']} at {e['time']}" for e in r["events"]]
        ans = f"On {hd} you have " + (items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]) + "."
        check = {"regex": [[p] for _, _, p in EVENTS[iso]]}
    else:
        ans = rng.choice([f"You have nothing scheduled on {hd}.", f"Your calendar is free on {hd}."])
        check = {"regex": [[r"\bno\b", r"nothing", r"\bfree\b", r"\bnot\b", r"\bclear\b", r"\bempty\b", r"\bnone\b",
                            r"n't\b[^.]*\bany\b"]]}            # "I don't have any events ...", "I couldn't find any ..."
    check["weekday"] = datetime.date.fromisoformat(iso).strftime("%A").lower()   # if the answer names a weekday, it must be this one
    return build(split, "calendar", rng.choice(C_T[split]).format(on=on, fr=fr), [([call], [r])], ans, check)

# ---------------- unit conversion ----------------
UNIT_WORDS = {"kilometers": ["km", "kilometers", "kilometres"], "meters": ["meters", "metres", "m"], "centimeters": ["cm", "centimeters"],
              "miles": ["miles", "mi"], "feet": ["feet", "ft"], "inches": ["inches"], "kilograms": ["kg", "kilograms", "kilos"],
              "grams": ["grams", "g"], "pounds": ["pounds", "lbs", "lb"], "ounces": ["ounces", "oz"],
              "liters": ["liters", "litres", "L"], "milliliters": ["ml", "milliliters"], "gallons": ["gallons", "gal"],
              "celsius": ["degrees Celsius", "°C", "Celsius"], "fahrenheit": ["degrees Fahrenheit", "°F", "Fahrenheit"]}
PAIRS = [("kilometers", "miles"), ("miles", "kilometers"), ("meters", "feet"), ("feet", "meters"), ("centimeters", "inches"),
         ("inches", "centimeters"), ("kilograms", "pounds"), ("pounds", "kilograms"), ("grams", "ounces"), ("ounces", "grams"),
         ("liters", "gallons"), ("gallons", "liters"), ("milliliters", "liters"), ("kilometers", "meters"),
         ("celsius", "fahrenheit"), ("fahrenheit", "celsius")]
CV_T = {"train": ["Convert {v} {f} to {t}.", "How many {t} is {v} {f}?", "What is {v} {f} in {t}?", "{v} {f} to {t}",
                  "Please change {v} {f} into {t}.", "I have {v} {f}. How much is that in {t}?"],
        "test": ["Express {v} {f} in {t}.", "If I've got {v} {f}, what would that be in {t}?", "Turn {v} {f} into {t} for me.",
                 "{v} {f} equals how many {t}?", "Quick conversion, please: {v} {f} to {t}."]}

def gen_convert(rng, split):
    f, t = rng.choice(PAIRS)
    if f == "celsius":
        v = rng.randint(-10, 40)
    elif f == "fahrenheit":
        v = rng.randint(20, 104)
    else:
        v = rng.randint(1, 300) if rng.random() < 0.7 else round(rng.uniform(0.5, 100), 1)
    call, r = run("convert_units", value=v, from_unit=f, to_unit=t)
    req = rng.choice(CV_T[split]).format(v=fmt_num(v), f=rng.choice(UNIT_WORDS[f]), t=rng.choice(UNIT_WORDS[t]))
    ans = f"{fmt_num(v)} {f} is {fmt_num(r['result'])} {t}."
    return build(split, "convert", req, [([call], [r])], ans, {"numbers": [r["result"]]})

# ---------------- contacts ----------------
CT_T = {"train": [("What's {first}'s phone number?", "phone"), ("Give me {first}'s email address.", "email"),
                  ("Can you find {full}'s number?", "phone"), ("Where does {first} live?", "city"),
                  ("Look up {first} in my contacts.", "all"), ("I need to email {first}. What's the address?", "email"),
                  ("How can I call {full}?", "phone")],
        "test": [("Fetch the contact details for {full}.", "all"), ("What number should I dial to reach {first}?", "phone"),
                 ("Which city is {first} based in?", "city"), ("What e-mail does {first} use?", "email"),
                 ("Do I have a phone number saved for {first}?", "phone"), ("{first}'s contact info, please.", "all")]}

def gen_contact(rng, split):
    tpl, field = rng.choice(CT_T[split])
    c = rng.choice(CONTACTS)
    name = c["name"] if "{full}" in tpl else c["name"].split()[0]
    call, r = run("find_contact", name=name)
    ans = {"phone": f"{r['name']}'s phone number is {r['phone']}.", "email": f"{r['name']}'s email address is {r['email']}.",
           "city": f"{r['name']} lives in {r['city']}.",
           "all": f"{r['name']}: phone {r['phone']}, email {r['email']}, lives in {r['city']}."}[field]
    check = {"phone": {"digits": [r["phone"]]}, "all": {"digits": [r["phone"]]}, "email": {"regex": [[re.escape(r["email"])]]},
             "city": {"regex": [[re.escape(r["city"].lower())]]}}[field]
    return build(split, "contact", tpl.format(first=c["name"].split()[0], full=c["name"]), [([call], [r])], ans, check)

# ---------------- chains: contact -> weather ----------------
CH_T = {"train": [("What's the weather like where {first} lives?", "tc"), ("Is it raining where {first} is?", "c"),
                  ("How hot is it in {first}'s city right now?", "t"), ("Check the weather in {first}'s hometown.", "tc")],
        "test": [("I'm about to call {first}. What's the weather over there?", "tc"),
                 ("What temperature is it in the city where {full} lives?", "t"), ("Should {first} take an umbrella today?", "c")]}

def gen_chain(rng, split):
    tpl, need = rng.choice(CH_T[split])
    c = rng.choice(CONTACTS)
    name = c["name"] if "{full}" in tpl else c["name"].split()[0]
    call1, r1 = run("find_contact", name=name)
    call2, r2 = run("get_weather", city=r1["city"])
    ans = f"{r1['name']} lives in {r2['city']}, where it's currently {r2['temperature']}°C and {r2['condition']}."
    return build(split, "chain", tpl.format(first=c["name"].split()[0], full=c["name"]), [([call1], [r1]), ([call2], [r2])],
                 ans, weather_check([r2], need))

# ---------------- parallel: two cities ----------------
PA_T = {"train": ["Compare the weather in {c1} and {c2}.", "What's the temperature in {c1} and in {c2}?", "Weather for {c1} and {c2}, please."],
        "test": ["Which is warmer right now, {c1} or {c2}?", "How do {c1} and {c2} compare weather-wise today?"]}

def gen_parallel(rng, split):
    c1, c2 = rng.sample(list(WEATHER), 2)
    (k1, r1), (k2, r2) = run("get_weather", city=c1), run("get_weather", city=c2)
    ans = (f"In {c1} it's {r1['temperature']}°C and {r1['condition']}, while in {c2} it's "
           f"{r2['temperature']}°C and {r2['condition']}.")
    return build(split, "parallel", rng.choice(PA_T[split]).format(c1=c1, c2=c2), [([k1, k2], [r1, r2])], ans,
                 {"numbers": [r1["temperature"], r2["temperature"]]})

# ---------------- no tool needed ----------------
CAPABILITY = "I'm your personal assistant. I can check the weather, look up your calendar and contacts, and convert units."
NONE_TRAIN_QA = [  # (request, answer)
    ("Hi!", "Hello! How can I help you today?"), ("Hello", "Hi there! What can I do for you?"),
    ("Good morning!", "Good morning! How can I help?"), ("How are you?", "I'm doing well, thanks for asking! How can I help you?"),
    ("Thanks!", "You're welcome!"), ("Thank you so much.", "You're very welcome. Let me know if you need anything else."),
    ("That's all, thanks.", "Happy to help. Have a great day!"), ("Bye!", "Goodbye! Have a great day."),
    ("What can you do?", CAPABILITY), ("Who are you?", CAPABILITY), ("What tools do you have?", CAPABILITY),
    ("How can you help me?", CAPABILITY),
    ("How many days are in a week?", "There are 7 days in a week."),
    ("How many days are there in one week?", "One week has 7 days."),
    ("How many minutes are in an hour?", "There are 60 minutes in an hour."),
    ("How many hours are in a day?", "There are 24 hours in a day."),
    ("What does humidity mean?", "Humidity is the amount of water vapour in the air."),
    ("Explain the word humidity.", "Humidity means how much water vapour the air holds."),
    ("Why is the sky blue?", "Air molecules scatter blue sunlight more than other colours, so the sky looks blue."),
    ("What causes rain?", "Rain forms when water vapour in clouds condenses into droplets heavy enough to fall."),
    ("What is a leap year?", "A leap year has 366 days, with an extra day on 29 February."),
    ("Who painted the Mona Lisa?", "Leonardo da Vinci painted the Mona Lisa."),
    ("What is the largest planet in our solar system?", "Jupiter is the largest planet in our solar system."),
    ("Which planet is the biggest?", "Jupiter is the biggest planet in the solar system."),
    ("Tell me a joke about calendars.", "Why was the calendar so popular? Because it had a lot of dates!"),
    ("Give me a tip for staying productive.", "Break big tasks into small steps and start with the hardest one."),
    ("At what temperature does water freeze in Celsius?", "Water freezes at 0 °C."),
    ("How many continents are there?", "There are seven continents."),
    ("What is the chemical symbol for gold?", "The chemical symbol for gold is Au."),
    ("Suggest a name for a pet cat.", "How about Luna? It's short, sweet and easy to call."),
    ("What's the opposite of hot?", "The opposite of hot is cold."),
    ("What do people use a calendar for?", "A calendar helps people keep track of dates, events and appointments."),
    ("Is a kilometre longer than a mile?", "No, a mile is longer than a kilometre."),
    ("Write a short sentence about the moon.", "The moon glows softly over the quiet sea."),
]
CAPITALS_TRAIN = [("France", "Paris"), ("Japan", "Tokyo"), ("Italy", "Rome"), ("Germany", "Berlin"), ("Spain", "Madrid"),
                  ("Egypt", "Cairo"), ("Canada", "Ottawa"), ("China", "Beijing"), ("Russia", "Moscow"), ("Greece", "Athens"),
                  ("Portugal", "Lisbon"), ("Norway", "Oslo"), ("Ireland", "Dublin"), ("Austria", "Vienna"),
                  ("South Korea", "Seoul"), ("Thailand", "Bangkok"), ("Peru", "Lima"), ("Kenya", "Nairobi")]
CAP_T_TRAIN = ["What is the capital of {c}?", "What's the capital city of {c}?", "Name the capital of {c}."]
ARITH_TRAIN = [("What is {a} plus {b}?", "{a} plus {b} is {r}.", lambda a, b: a + b),
               ("What's {a} times {b}?", "{a} times {b} is {r}.", lambda a, b: a * b),
               ("Calculate {a} minus {b}.", "{a} minus {b} is {r}.", lambda a, b: a - b)]

NONE_TEST = [  # (request, answer check); every check is something a 0.5B model can know without tools
    ("Which city is the capital of Sweden?", {"regex": [["stockholm"]]}), ("Do you know Poland's capital?", {"regex": [["warsaw"]]}),
    ("Which city is the capital of Argentina?", {"regex": [["buenos aires"]]}), ("Do you know Mexico's capital?", {"regex": [["mexico city"]]}),
    ("Which city is the capital of Turkey?", {"regex": [["ankara"]]}), ("Do you know Vietnam's capital?", {"regex": [["hanoi"]]}),
    ("Which city is the capital of the Netherlands?", {"regex": [["amsterdam"]]}),
    ("Add 37 and 48.", {"numbers": [85]}), ("What do you get if you multiply 7 by 8?", {"numbers": [56]}),
    ("Subtract 19 from 64.", {"numbers": [45]}), ("Add 125 and 250.", {"numbers": [375]}),
    ("What do you get if you multiply 9 by 6?", {"numbers": [54]}),
    ("Hey, good evening!", {}), ("Cheers, that's everything I needed.", {}), ("Hiya, how's it going?", {}),
    ("Many thanks for the help!", {}), ("What sorts of things can you help me with?", {}), ("Tell me a bit about yourself.", {}),
    ("How many seconds are there in a minute?", {"numbers": [60]}),
    ("At what temperature does water boil, in Celsius?", {"numbers": [100]}),
    ("Who wrote Romeo and Juliet?", {"regex": [["shakespeare"]]}),
    ("Which planet is closest to the sun?", {"regex": [["mercury"]]}),
    ("Is a tomato a fruit or a vegetable?", {"regex": [["fruit"]]}),
    ("How long is a marathon in kilometres?", {"numbers": [42.195]}),
    ("Write a two-line poem about the sea.", {}), ("Explain what fog is in one sentence.", {}),
    ("What's the weather usually like in a desert?", {}),
]

def none_items(rng, split):
    items = []
    if split == "train":
        for q, a in NONE_TRAIN_QA:
            items.append(build(split, "none", q, [], a, {}))
        for c, cap in CAPITALS_TRAIN:
            for tpl in CAP_T_TRAIN:
                items.append(build(split, "none", tpl.format(c=c), [], f"The capital of {c} is {cap}.", {"regex": [[cap.lower()]]}))
        seen = set()
        while len(seen) < 60:
            tq, ta, fn = rng.choice(ARITH_TRAIN)
            a, b = (rng.randint(2, 12), rng.randint(2, 12)) if "times" in tq else (rng.randint(10, 99), rng.randint(2, 60))
            q = tq.format(a=a, b=b)
            if q not in seen:
                seen.add(q)
                items.append(build(split, "none", q, [], ta.format(a=a, b=b, r=fn(a, b)), {"numbers": [fn(a, b)]}))
    else:
        for q, check in NONE_TEST:
            items.append(build(split, "none", q, [], "", check))
    return items

GEN = {"weather": gen_weather, "calendar": gen_calendar, "convert": gen_convert, "contact": gen_contact,
       "chain": gen_chain, "parallel": gen_parallel}
TRAIN_COUNTS = {"weather": 110, "calendar": 110, "convert": 110, "contact": 110, "chain": 60, "parallel": 40}
TEST_COUNTS = {"weather": 15, "calendar": 15, "convert": 15, "contact": 15, "chain": 8, "parallel": 5}   # + 27 no-tool = 100

def make_split(split, counts, seed):
    rng, items, seen = random.Random(seed), [], set()
    for cat, n in counts.items():
        got, tries = 0, 0
        while got < n and tries < 200 * n:
            tries += 1
            it = GEN[cat](rng, split)
            if it["request"] not in seen:
                seen.add(it["request"]); items.append(it); got += 1
        assert got == n, f"could not make {n} distinct {split} {cat} requests"
    items += none_items(rng, split)
    rng.shuffle(items)
    for k, it in enumerate(items):
        it["id"] = f"{split}-{k:03d}"
    return items

TRAIN = make_split("train", TRAIN_COUNTS, SEED)
TEST = make_split("test", TEST_COUNTS, SEED + 1)
assert not {i["request"] for i in TRAIN} & {i["request"] for i in TEST}, "a test request also appears in training"
assert len(TEST) == 100
for name, items in (("train", TRAIN), ("test", TEST)):
    with open(f"{DATA_DIR}/{name}.jsonl", "w") as f:
        for it in items:
            f.write(json.dumps(it) + "\n")

cats = ["weather", "calendar", "convert", "contact", "chain", "parallel", "none"]
print(f"{'':6s}" + "".join(f"{c:>10s}" for c in cats) + f"{'total':>8s}")
for name, items in (("train", TRAIN), ("test", TEST)):
    cnt = collections.Counter(i["category"] for i in items)
    print(f"{name:6s}" + "".join(f"{cnt[c]:10d}" for c in cats) + f"{len(items):8d}")
print("\nexamples (test):")
for it in TEST[:6]:
    print(f"  [{it['category']:8s}] {it['request']:55s} -> {it['calls']}")

         weather  calendar   convert   contact     chain  parallel      none   total
train        110       110       110       110        60        40       148     688
test          15        15        15        15         8         5        27     100

examples (test):
  [none    ] Is a tomato a fruit or a vegetable?                     -> []
  [calendar] Remind me what I've got going on tomorrow.              -> [[{'name': 'get_events', 'arguments': {'date': '2026-03-11'}}]]
  [none    ] Write a two-line poem about the sea.                    -> []
  [convert ] Express 295 km in m.                                    -> [[{'name': 'convert_units', 'arguments': {'value': 295, 'from_unit': 'kilometers', 'to_unit': 'meters'}}]]
  [contact ] Which city is Emma based in?                            -> [[{'name': 'find_contact', 'arguments': {'name': 'Emma'}}]]
  [none    ] Cheers, that's everything I needed.                     -> []


## 4. Metrics
For every test request the agent's whole trajectory is scored:

| Metric | Over | Definition |
|---|---|---|
| **Call validity** | 73 tool requests | the model made ≥1 call and **every** call parses in the expected format: a `<tool_call>` block with JSON `{"name", "arguments"}`, a known tool, required arguments present, correct JSON types |
| **Call accuracy** | 73 tool requests | valid **and** the calls made are exactly the right tools with the right arguments (the gold calls as a multiset; small spelling variants like "km"/"kilometers" or "sarah"/"Sarah Collins" count as equal) |
| **Abstention** | 27 no-tool requests | the model answered directly, with no tool call |
| **Task success** | all 100 | the final answer is correct end to end: it contains the facts the tools returned (or the known answer), and no-tool requests were answered without a tool |

A request **passes** if it is fully right (accurate calls and correct answer, or abstained and correct answer). Every failure gets one category (first matching rule): *wrong format*, *no call* (answered from memory), *wrong tool*, *missing call*, *extra/repeated call*, *wrong argument*, *step limit*, *invented wrong detail* (the right facts plus a wrong one that no tool returned, e.g. a wrong weekday), *ignored/misreported tool result*; for no-tool requests: *unnecessary call* or *wrong direct answer*.

Two checks were added after reading the first run's outputs; they apply to all models alike. (1) A free day may also be reported as "I don't have any events" or "I couldn't find any events". (2) The tools never return a weekday, so if an answer names one, it must be the right one.

In [6]:
NUM_RE = re.compile(r"-?\d+(?:,\d{3})*(?:\.\d+)?")
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

def number_matches(n, x):
    # n (found in the answer) equals x up to rounding to 1-2 decimals, or to an integer for |x| >= 10
    return (any(abs(n - round(x, k)) < 1e-6 for k in (1, 2, 6))) or (abs(x) >= 10 and abs(n - round(x)) < 1e-6)

def check_answer(text, spec):
    if not text or not text.strip():
        return False
    t = text.lower().replace("−", "-")
    if any(not any(re.search(p, t) for p in group) for group in spec.get("regex", [])):
        return False
    nums = [float(x.replace(",", "")) for x in NUM_RE.findall(t)]
    if any(not any(number_matches(n, x) for n in nums) for x in spec.get("numbers", [])):
        return False
    if "weekday" in spec:      # the tools never return a weekday, so a named weekday must at least be the right one
        named = {d for d in WEEKDAYS if re.search(rf"\b{d}\b", t)}
        if named and named != {spec["weekday"]}:
            return False
    digits = re.sub(r"\D", "", t)
    return all(re.sub(r"\D", "", s) in digits for s in spec.get("digits", []))

def classify(item, t, made, gold, fmt_err):
    if not item["needs_tool"]:
        return "unnecessary call" if made else ("wrong format" if fmt_err else "wrong direct answer")
    if fmt_err or (made and not all(is_valid_call(c) for c in made)):
        return "wrong format"
    if not made:
        return "no call (answered directly)"
    mn, gn = collections.Counter(c["name"] for c in made), collections.Counter(c["name"] for c in gold)
    if set(mn) - set(gn):
        return "wrong tool"
    mk, gk = collections.Counter(norm_call(c) for c in made), collections.Counter(norm_call(c) for c in gold)
    if mk != gk:
        if not (gk - mk) and (mk - gk):
            return "extra/repeated call"
        if not (mk - gk) and (gk - mk):
            return "missing call"
        return "wrong argument" if mn == gn or not (mn - gn) else "extra/repeated call"
    if t["stop"] == "max_steps":
        return "step limit"
    if check_answer(t["final"], {k: v for k, v in item["check"].items() if k != "weekday"}):
        return "invented wrong detail"      # right facts, plus a wrong one the tool never gave (a weekday)
    return "ignored/misreported tool result"

def score_item(item, t):
    made = [c for s in t["steps"] for c in s["calls"]]
    fmt_err = any(s["format_error"] for s in t["steps"])
    gold = [c for step in item["calls"] for c in step]
    r = {"id": item["id"], "category": item["category"], "needs_tool": item["needs_tool"], "n_calls": len(made),
         "format_error": fmt_err, "stop": t["stop"]}
    r["answer_ok"] = t["stop"] != "max_steps" and check_answer(t["final"], item["check"])
    if item["needs_tool"]:
        r["valid"] = bool(made) and not fmt_err and all(is_valid_call(c) for c in made)
        r["accurate"] = r["valid"] and collections.Counter(map(norm_call, made)) == collections.Counter(map(norm_call, gold))
        r["success"] = r["answer_ok"]
        r["pass"] = r["accurate"] and r["success"]
    else:
        r["abstained"] = not made and not fmt_err
        r["success"] = r["abstained"] and r["answer_ok"]
        r["pass"] = r["success"]
    r["failure"] = None if r["pass"] else classify(item, t, made, gold, fmt_err)
    return r

def metrics(rows):
    tool = [r for r in rows if r["needs_tool"]]
    none = [r for r in rows if not r["needs_tool"]]
    m = lambda xs: 100 * float(np.mean(xs)) if xs else float("nan")
    return {"call_validity": m([r["valid"] for r in tool]), "call_accuracy": m([r["accurate"] for r in tool]),
            "abstention": m([r["abstained"] for r in none]), "task_success": m([r["success"] for r in rows]),
            "fully_correct": m([r["pass"] for r in rows]), "n_tool": len(tool), "n_none": len(none)}

def paired_ci(a, b, n=10000, seed=0):
    # 95% bootstrap CI of mean(b) - mean(a) over the same requests
    a, b = np.asarray(a, float), np.asarray(b, float)
    idx = np.random.default_rng(seed).integers(0, len(a), size=(n, len(a)))
    d = (b[idx] - a[idx]).mean(1)
    return 100 * (b.mean() - a.mean()), 100 * np.percentile(d, 2.5), 100 * np.percentile(d, 97.5)

TEST_EVAL = TEST if EVAL_LIMIT is None else TEST[:EVAL_LIMIT]

def evaluate(name, model, system_prompt=SYSTEM_PROMPT):
    path = f"{RES_DIR}/traj_{name}.json"
    if os.path.exists(path) and not RERUN and EVAL_LIMIT is None:
        traj = json.load(open(path))
        print(f"{name}: loaded saved trajectories from Drive")
    else:
        t0 = time.time()
        traj = run_agent(model, [it["request"] for it in TEST_EVAL], system_prompt, verbose=True)
        print(f"{name}: {len(traj)} requests in {time.time() - t0:.0f}s")
        json.dump(traj, open(path, "w"), indent=1)
    rows = [score_item(it, t) for it, t in zip(TEST_EVAL, traj)]
    mt = metrics(rows)
    print("   " + " | ".join(f"{k} {v:.1f}" for k, v in mt.items() if not k.startswith("n_")))
    return traj, rows

## 5. Baseline: the untouched model
The model is evaluated twice, both times without finetuning:
* **Baseline:** the plain system prompt (one sentence plus the tool schemas the chat template adds).
* **Baseline + better prompt:** the same, plus explicit rules on when to call a tool and how to format dates, and one example call. This shows how much prompting alone can do.

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)   # fp32 weights, fp16 autocast
GEN_CFG = GenerationConfig(do_sample=False, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id,
                           eos_token_id=[tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")])
model.generation_config = GEN_CFG      # greedy decoding, no repetition penalty (replaces Qwen's default sampling settings)
print(f"{MODEL_ID}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

BETTER_PROMPT = SYSTEM_PROMPT + '''

Rules:
- Call a tool only when the request needs the user's live or personal data (weather, calendar, contacts) or a unit conversion. For greetings, general knowledge, arithmetic or writing, answer directly without any tool.
- get_events needs the date as YYYY-MM-DD (for example, tomorrow is 2026-03-11).
- If one tool's result is needed for another call (for example a contact's city, then its weather), make the calls one after the other.
- After a tool returns, answer in one short sentence using the exact values it returned.
Example: for "What's the weather in Oslo?" reply with
<tool_call>
{"name": "get_weather", "arguments": {"city": "Oslo"}}
</tool_call>'''

print("A few requests through the agent loop:\n")
for t in run_agent(model, ["Do I need an umbrella in Kandy today?", "Which city is the capital of Sweden?",
                           "What temperature is it in the city where Sarah Collins lives?"]):
    show(t)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-0.5B-Instruct: 494M parameters
A few requests through the agent loop:

USER: Do I need an umbrella in Kandy today?
  MODEL: <tool_call> {"name": "get_weather", "arguments": {"city": "Kandy", "unit": "fahrenheit"}} </tool_call>
  TOOL : {"city": "Kandy", "temperature": 75, "unit": "fahrenheit", "condition": "light rain", "humidity_percent": 85}
  MODEL: Today in Kandy, the temperature is 75°F with light rain. The humidity is at 85%. It's a good idea to bring an umbrella for the light rain.
  -> stop: answer

USER: Which city is the capital of Sweden?
  MODEL: <tool_call> {"name": "get_weather", "arguments": {"city": "Stockholm", "unit": "fahrenheit"}} </tool_call>
  TOOL : {"error": "No weather data for 'Stockholm'."}
  MODEL: The current weather in Stockholm is 13°C and it's partly cloudy.
  -> stop: answer

USER: What temperature is it in the city where Sarah Collins lives?
  MODEL: <tool_call> {"name": "get_weather", "arguments": {"city": "Sarah Collins", "unit": "fahren

In [8]:
traj_base, rows_base = evaluate("baseline", model, SYSTEM_PROMPT)
traj_prompt, rows_prompt = evaluate("baseline_better_prompt", model, BETTER_PROMPT)

baseline: loaded saved trajectories from Drive
   call_validity 97.3 | call_accuracy 67.1 | abstention 29.6 | task_success 58.0 | fully_correct 57.0
baseline_better_prompt: loaded saved trajectories from Drive
   call_validity 95.9 | call_accuracy 75.3 | abstention 29.6 | task_success 62.0 | fully_correct 62.0


## 6. LoRA finetuning
* **What is trained:** LoRA adapters (rank 16, α = 32, dropout 0.05) on every attention and MLP projection of the language model; the base weights stay frozen.
* **Loss on the assistant turns only:** each training conversation is rendered with the same chat template and tool schemas as at test time, and only the tokens the assistant writes (its tool calls, its final answer and the closing `<|im_end|>`) get a label. The system prompt, the user request and the tool results get label −100. The check below prints exactly what is trained on.
* 2 epochs, AdamW, learning rate 2e-4 with 5% warm-up then linear decay, effective batch 8 (2 conversations × 4 accumulation steps), fp16 mixed precision. The adapter is saved to Drive.
* **Memory:** the output scores over the 151,936-word vocabulary are computed only at the labelled (assistant) positions. The loss is identical, and the T4's 15 GB is enough.

In [9]:
from peft import LoraConfig, get_peft_model, PeftModel

ASSIST_RE = re.compile(r"<\|im_start\|>assistant\n(.*?<\|im_end\|>)", re.S)

def encode_example(messages):
    text = tok.apply_chat_template(messages, tools=TOOLS, tokenize=False)
    enc = tok(text, return_offsets_mapping=True, add_special_tokens=False)
    spans = [(m.start(1), m.end(1)) for m in ASSIST_RE.finditer(text)]
    labels = [tid if any(s >= a and e <= b for a, b in spans) else -100
              for tid, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"input_ids": enc["input_ids"][:MAX_LEN], "labels": labels[:MAX_LEN], "truncated": len(labels) > MAX_LEN}

train_enc = [encode_example(it["messages"]) for it in TRAIN]
lens = [len(e["input_ids"]) for e in train_enc]
print(f"{len(train_enc)} training conversations | tokens: mean {np.mean(lens):.0f}, max {max(lens)} | "
      f"truncated {sum(e['truncated'] for e in train_enc)} | trained-on tokens per example: "
      f"{np.mean([sum(l != -100 for l in e['labels']) for e in train_enc]):.0f}")

# check the masking on a chain example: only the assistant's own turns must appear
ex = next(e for e, it in zip(train_enc, TRAIN) if it["category"] == "chain")
pieces, cur = [], []
for tid, lab in zip(ex["input_ids"], ex["labels"]):
    if lab != -100:
        cur.append(tid)
    elif cur:
        pieces.append(tok.decode(cur)); cur = []
print("trained on:", [p for p in pieces + ([tok.decode(cur)] if cur else [])])

688 training conversations | tokens: mean 720, max 824 | truncated 0 | trained-on tokens per example: 42
trained on: ['<tool_call>\n{"name": "find_contact", "arguments": {"name": "Tharindu"}}\n</tool_call><|im_end|>', '<tool_call>\n{"name": "get_weather", "arguments": {"city": "Colombo"}}\n</tool_call><|im_end|>', "Tharindu Jayasuriya lives in Colombo, where it's currently 31°C and partly cloudy.<|im_end|>"]


In [10]:
def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    attn = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"]); labels[i, :n] = torch.tensor(b["labels"]); attn[i, :n] = 1
    return {"input_ids": ids, "labels": labels, "attention_mask": attn}

if isinstance(globals().get("lora_model"), PeftModel):   # this cell was run before: take the old adapter off first
    model = lora_model.unload(); del lora_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

if os.path.exists(f"{ADAPTER_DIR}/adapter_config.json") and not RERUN and MAX_TRAIN_STEPS is None:
    lora_model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    train_log = json.load(open(f"{RES_DIR}/train_log.json"))
    print("loaded the trained adapter from Drive")
else:
    set_seed(SEED)
    lora_model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                                  target_modules=LORA_TARGETS, task_type="CAUSAL_LM"))
    lora_model.print_trainable_parameters()
    inner = lora_model.get_base_model()      # the Qwen model with the LoRA layers inside

    def train_loss(batch):
        # Same value as lora_model(**batch).loss (checked: equal to 1e-6), but the scores over the 151,936-word
        # vocabulary are computed only where there is a label (the assistant's tokens, about 6% of positions)
        # instead of at every position. That saves several GB of GPU memory.
        h = inner.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state
        target = batch["labels"][:, 1:]              # token t+1 is predicted from position t
        keep = target != -100
        logits = inner.lm_head(h[:, :-1][keep]).float()
        return torch.nn.functional.cross_entropy(logits, target[keep])

    params = [p for p in lora_model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    loader = torch.utils.data.DataLoader(train_enc, batch_size=TRAIN_BATCH, shuffle=True, collate_fn=collate,
                                         generator=torch.Generator().manual_seed(SEED))
    total = EPOCHS * math.ceil(len(loader) / GRAD_ACCUM)
    if MAX_TRAIN_STEPS:
        total = min(total, MAX_TRAIN_STEPS)
    warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    lora_model.train()
    step, micro, losses, t0 = 0, 0, [], time.time()
    train_log = {"steps": [], "loss": []}
    while step < total:
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = train_loss(batch) / GRAD_ACCUM
            scaler.scale(loss).backward()
            losses.append(loss.item() * GRAD_ACCUM); micro += 1
            if micro % GRAD_ACCUM:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            step += 1
            train_log["steps"].append(step); train_log["loss"].append(float(np.mean(losses[-GRAD_ACCUM:])))
            if step % 10 == 0 or step == total:
                print(f"   step {step:3d}/{total} | loss {np.mean(losses[-10 * GRAD_ACCUM:]):.4f} | {time.time() - t0:.0f}s", flush=True)
            if step >= total:
                break
    train_log["train_time_s"] = round(time.time() - t0)
    if DEVICE == "cuda":
        train_log["peak_gpu_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
        print(f"   peak GPU memory during training: {train_log['peak_gpu_gb']} GB")
    del opt, scaler, batch, loss
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if MAX_TRAIN_STEPS is None:
        lora_model.save_pretrained(ADAPTER_DIR)
        json.dump(train_log, open(f"{RES_DIR}/train_log.json", "w"))
        print("adapter saved to", ADAPTER_DIR)

loaded the trained adapter from Drive


In [11]:
traj_lora, rows_lora = evaluate("lora", lora_model, SYSTEM_PROMPT)
print()
for t in traj_lora[:3]:
    show(t)

lora: loaded saved trajectories from Drive
   call_validity 100.0 | call_accuracy 93.2 | abstention 96.3 | task_success 85.0 | fully_correct 85.0

USER: Is a tomato a fruit or a vegetable?
  MODEL: Tomatoes are a fruit.
  -> stop: answer

USER: Remind me what I've got going on tomorrow.
  MODEL: <tool_call> {"name": "get_events", "arguments": {"date": "2026-03-11"}} </tool_call>
  TOOL : {"date": "2026-03-11", "events": [{"time": "11:00", "title": "Project review with Sarah"}]}
  MODEL: On Wednesday, 11 March you have Project review with Sarah at 11:00.
  -> stop: answer

USER: Write a two-line poem about the sea.
  MODEL: The waves crash against the shore,   The sea whispers secrets to the shore.
  -> stop: answer



## 7. Results and failure analysis
* **The table:** the four required metrics for each model. The last column is LoRA minus baseline, with a paired bootstrap 95% CI over the same test requests.
* **Failures:** one category per failed request. A cross-table then shows which baseline failures LoRA **fixed**, which it did **not fix**, and which new failures it **introduced**.
* **Examples:** at least 10 failures from each model are printed, spread across the categories.

In [12]:
MODELS = {"baseline": rows_base, "baseline + better prompt": rows_prompt, "LoRA": rows_lora}
M = {k: metrics(v) for k, v in MODELS.items()}
KEYS = [("call_validity", "Call validity", "valid", True), ("call_accuracy", "Call accuracy", "accurate", True),
        ("abstention", "Abstention", "abstained", False), ("task_success", "Task success", "success", None),
        ("fully_correct", "Fully correct", "pass", None)]
def diff_table(models, keys=KEYS):
    # one row per metric; paired bootstrap 95% CIs over the same test requests
    mt = {k: metrics(v) for k, v in models.items()}
    base, prm, lra = models["baseline"], models["baseline + better prompt"], models["LoRA"]
    print(f"{'metric (%)':15s}{'baseline':>10s}{'+ prompt':>10s}{'LoRA':>8s}   {'LoRA - baseline':>22s}   {'prompt - baseline':>22s}")
    for key, label, field, tool in keys:
        sel = [i for i, r in enumerate(base) if tool is None or r["needs_tool"] == tool]
        ci = lambda a, b: paired_ci([a[i][field] for i in sel], [b[i][field] for i in sel]) if sel else (np.nan,) * 3
        f = lambda d: f"{d[0]:+6.1f} [{d[1]:+6.1f}, {d[2]:+6.1f}]"
        print(f"{label:15s}{mt['baseline'][key]:10.1f}{mt['baseline + better prompt'][key]:10.1f}{mt['LoRA'][key]:8.1f}"
              f"   {f(ci(base, lra)):>22s}   {f(ci(base, prm)):>22s}")

diff_table(MODELS)
print(f"(validity/accuracy over {M['LoRA']['n_tool']} tool requests, abstention over {M['LoRA']['n_none']} no-tool requests, the rest over all)")

print("\nTask success by category:")
cats = ["weather", "calendar", "convert", "contact", "chain", "parallel", "none"]
print(f"{'':26s}" + "".join(f"{c:>10s}" for c in cats))
for k, rows in MODELS.items():
    print(f"{k:26s}" + "".join(f"{100 * np.mean([r['success'] for r in rows if r['category'] == c]):10.0f}" for c in cats))

print("\nFailure categories (number of requests):")
fc = {k: collections.Counter(r["failure"] for r in rows if r["failure"]) for k, rows in MODELS.items()}
allc = sorted(set().union(*fc.values()), key=lambda c: -fc["baseline"][c])
print(f"{'':34s}" + "".join(f"{k:>26s}" for k in MODELS))
for c in allc:
    print(f"{c:34s}" + "".join(f"{fc[k][c]:26d}" for k in MODELS))
print(f"{'total failures':34s}" + "".join(f"{sum(fc[k].values()):26d}" for k in MODELS))

print("\nBaseline -> LoRA, per request:")
trans = collections.defaultdict(collections.Counter)
for b, l in zip(rows_base, rows_lora):
    if b["failure"] and l["pass"]:
        trans[b["failure"]]["fixed"] += 1
    elif b["failure"]:
        trans[b["failure"]]["not fixed (same)" if l["failure"] == b["failure"] else "not fixed (other failure)"] += 1
    elif l["failure"]:
        trans[l["failure"]]["introduced"] += 1
cols = ["fixed", "not fixed (same)", "not fixed (other failure)", "introduced"]
print(f"{'category':34s}" + "".join(f"{c:>28s}" for c in cols))
for c in sorted(trans, key=lambda c: -sum(trans[c].values())):
    print(f"{c:34s}" + "".join(f"{trans[c][x]:28d}" for x in cols))

summary = {"metrics": M, "failure_counts": {k: dict(v) for k, v in fc.items()},
           "transitions": {k: dict(v) for k, v in trans.items()},
           "sizes": {"train": len(TRAIN), "test": len(TEST_EVAL)}, "train_log": train_log,
           "settings": {"model": MODEL_ID, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
                        "targets": LORA_TARGETS, "lr": LR, "epochs": EPOCHS, "batch": TRAIN_BATCH * GRAD_ACCUM, "seed": SEED}}
json.dump(summary, open(f"{RES_DIR}/summary.json", "w"), indent=1)
json.dump({k: v for k, v in MODELS.items()}, open(f"{RES_DIR}/per_request_scores.json", "w"))

metric (%)       baseline  + prompt    LoRA          LoRA - baseline        prompt - baseline
Call validity        97.3      95.9   100.0     +2.7 [  +0.0,   +6.8]     -1.4 [  -6.8,   +4.1]
Call accuracy        67.1      75.3    93.2    +26.0 [ +16.4,  +35.6]     +8.2 [  -1.4,  +17.8]
Abstention           29.6      29.6    96.3    +66.7 [ +48.1,  +85.2]     +0.0 [ -11.1,  +11.1]
Task success         58.0      62.0    85.0    +27.0 [ +15.0,  +39.0]     +4.0 [  -4.0,  +12.0]
Fully correct        57.0      62.0    85.0    +28.0 [ +16.0,  +40.0]     +5.0 [  -2.0,  +13.0]
(validity/accuracy over 73 tool requests, abstention over 27 no-tool requests, the rest over all)

Task success by category:
                             weather  calendar   convert   contact     chain  parallel      none
baseline                          40        87       100        93        12        20        30
baseline + better prompt          80        87       100        80         0        60        26
LoRA      

### Robustness checks
* **The Celsius convention.** `get_weather` defaults to Celsius, and the scorer treats an unrequested `unit="fahrenheit"` as a wrong argument. The table is recomputed with that choice accepted: the call counts as right, and the answer is checked against the Fahrenheit values the tool returned.
* **What the hand-written prompt did.** It counts how often each model added Fahrenheit unasked, and how often it called `get_events` with 2026-03-11, the date used as an example in the better prompt, when the request was for a different day.
* **Invented weekdays.** It counts how often each model's answer named a weekday, and how often that weekday was wrong.

In [13]:
def lenient_row(it, t, r):
    # same score, except that an unrequested unit="fahrenheit" is accepted (call and answer)
    gold = [c for s in it["calls"] for c in s]
    if not it["needs_tool"] or {c["arguments"].get("unit", "celsius") for c in gold if c["name"] == "get_weather"} != {"celsius"}:
        return r
    made = [c for s in t["steps"] for c in s["calls"]]
    no_unit = lambda c: norm_call(c)[:2] if c.get("name") == "get_weather" and norm_call(c) else norm_call(c)
    acc = r["valid"] and collections.Counter(map(no_unit, made)) == collections.Counter(map(no_unit, gold))
    ok = r["answer_ok"]
    used_f = any(c["name"] == "get_weather" and canon_temp_unit(c["arguments"].get("unit", "celsius")) == "fahrenheit" for c in made)
    if used_f and not ok and t["stop"] != "max_steps":
        to_f = {get_weather(c["arguments"]["city"])["temperature"]: get_weather(c["arguments"]["city"], unit="fahrenheit")["temperature"]
                for c in gold if c["name"] == "get_weather"}
        ok = check_answer(t["final"], dict(it["check"], numbers=[to_f.get(x, x) for x in it["check"].get("numbers", [])]))
    return dict(r, accurate=acc, answer_ok=ok, success=ok, **{"pass": acc and ok})

TRAJ = {"baseline": traj_base, "baseline + better prompt": traj_prompt, "LoRA": traj_lora}
LENIENT = {k: [lenient_row(it, t, r) for it, t, r in zip(TEST_EVAL, TRAJ[k], MODELS[k])] for k in MODELS}
print("Unrequested Fahrenheit accepted:")
diff_table(LENIENT)

print("\nBehaviour counts:")
for k, traj in TRAJ.items():
    n_w = n_f = n_ex = n_day = n_bad = 0
    for it, t in zip(TEST_EVAL, traj):
        gold = [c for s in it["calls"] for c in s]
        made = [c for s in t["steps"] for c in s["calls"]]
        if any(c["name"] == "get_weather" for c in gold) and all(c["arguments"].get("unit", "celsius") == "celsius" for c in gold if c["name"] == "get_weather"):
            w = [c for c in made if c["name"] == "get_weather"]
            n_w += bool(w)
            n_f += any(canon_temp_unit(c["arguments"].get("unit", "celsius")) == "fahrenheit" for c in w)
        n_ex += any(c["name"] == "get_events" and c["arguments"].get("date") == "2026-03-11" for c in made) and \
                not any(g["arguments"].get("date") == "2026-03-11" for g in gold)
        named = {d for d in WEEKDAYS if re.search(rf"\b{d}\b", str(t["final"]).lower())}
        if named:
            n_day += 1
            n_bad += named != {it["check"].get("weekday")}
    print(f"  {k:26s} unrequested Fahrenheit {n_f}/{n_w} | get_events(2026-03-11) not asked for: {n_ex} | "
          f"answers naming a weekday: {n_day} (wrong: {n_bad})")
summary["lenient_metrics"] = {k: metrics(v) for k, v in LENIENT.items()}
json.dump(summary, open(f"{RES_DIR}/summary.json", "w"), indent=1)

Unrequested Fahrenheit accepted:
metric (%)       baseline  + prompt    LoRA          LoRA - baseline        prompt - baseline
Call validity        97.3      95.9   100.0     +2.7 [  +0.0,   +6.8]     -1.4 [  -6.8,   +4.1]
Call accuracy        84.9      79.5    93.2     +8.2 [  +0.0,  +16.4]     -5.5 [ -13.7,   +1.4]
Abstention           29.6      29.6    96.3    +66.7 [ +48.1,  +85.2]     +0.0 [ -11.1,  +11.1]
Task success         71.0      65.0    85.0    +14.0 [  +3.0,  +25.0]     -6.0 [ -12.0,   +0.0]
Fully correct        70.0      65.0    85.0    +15.0 [  +4.0,  +26.0]     -5.0 [ -11.0,   +1.0]

Behaviour counts:
  baseline                   unrequested Fahrenheit 18/23 | get_events(2026-03-11) not asked for: 0 | answers naming a weekday: 0 (wrong: 0)
  baseline + better prompt   unrequested Fahrenheit 3/22 | get_events(2026-03-11) not asked for: 5 | answers naming a weekday: 0 (wrong: 0)
  LoRA                       unrequested Fahrenheit 0/24 | get_events(2026-03-11) not asked f

In [14]:
def show_failures(name, rows, traj, n_min=12):
    # print at least n_min failures, taking them round-robin across categories so every category is represented
    by = collections.defaultdict(list)
    for it, r, t in zip(TEST_EVAL, rows, traj):
        if r["failure"]:
            by[r["failure"]].append((it, t))
    picked, k = [], 0
    while len(picked) < n_min and any(len(v) > k for v in by.values()):
        picked += [(c, v[k]) for c, v in by.items() if len(v) > k]; k += 1
    print(f"==================== {name}: {sum(map(len, by.values()))} failures, showing {len(picked)} ====================")
    for c, (it, t) in picked:
        print(f"[{c}]  gold calls: {[(g['name'], g['arguments']) for s in it['calls'] for g in s] or 'none'}")
        show(t)
        print(f"  final answer: {str(t['final'])[:200]!r}\n")

show_failures("BASELINE", rows_base, traj_base)
show_failures("LoRA", rows_lora, traj_lora)

==================== BASELINE: 43 failures, showing 12 ====================
[unnecessary call]  gold calls: none
USER: Is a tomato a fruit or a vegetable?
  MODEL: <tool_call> {"name": "get_weather", "arguments": {"city": "New York", "unit": "fahrenheit"}} </tool_call>
  TOOL : {"city": "New York", "temperature": 43, "unit": "fahrenheit", "condition": "windy", "humidity_percent": 48}
  MODEL: The temperature in New York is 43 degrees Fahrenheit, which is windy. The humidity is 48%.
  -> stop: answer

  final answer: 'The temperature in New York is 43 degrees Fahrenheit, which is windy. The humidity is 48%.'

[no call (answered directly)]  gold calls: [('find_contact', {'name': 'Emma'})]
USER: Which city is Emma based in?
  MODEL: To provide you with the information, I need to know Emma's name. Could you please tell me her name?
  -> stop: answer

  final answer: "To provide you with the information, I need to know Emma's name. Could you please tell me her name?"

[wrong argument]  gold